In [1]:
pip install langchain

In [2]:
pip install langchain-community langchain-text-splitters langchain-groq langchain-huggingface langchain-core faiss-cpu sentence-transformers python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 120.0 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
pip install groq

In [4]:
pip show langchain

Name: langchain
Version: 1.4.7
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [7]:

import os
import sys

# --------------------------------------------------
# 1. SET GROQ API KEY
# --------------------------------------------------

from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API")

print("Groq API key loaded successfully!")


# --------------------------------------------------
# 2. IMPORT LIBRARIES
# --------------------------------------------------

try:
    from langchain_community.document_loaders import TextLoader
    from langchain_text_splitters import CharacterTextSplitter
    from langchain_groq import ChatGroq
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_community.vectorstores import FAISS
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.runnables import RunnablePassthrough
    from langchain_core.output_parsers import StrOutputParser

    print("Libraries imported successfully.")

except ImportError as e:
    print(f"Error importing libraries: {e}")
    sys.exit(1)


# --------------------------------------------------
# 3. CREATE COMPANY POLICY FILE
# --------------------------------------------------

policy_text = """
At TechStream, we are committed to providing the best electronics
to our customers.

RETURN POLICY:
We offer a 14-day return policy for all electronics, starting from
the date of delivery.
However, there is a special exception for laptops: Laptops must be
returned unopened with the original factory seal intact.
If the seal on a laptop box is broken, we cannot accept the return
due to security and data privacy reasons.

SHIPPING INFO:
Shipping is free for all orders over $50 within the continental
United States.
For orders under $50, a flat shipping fee of $5.99 applies.
International shipping is currently not available.

SUPPORT HOURS:
For technical support, our team is available Monday to Friday,
from 9:00 AM to 5:00 PM EST.
We are closed on weekends and federal holidays.
"""

with open("company_policy.txt", "w", encoding="utf-8") as f:
    f.write(policy_text)

print("Knowledge base created: company_policy.txt")


# --------------------------------------------------
# 4. FORMAT RETRIEVED DOCUMENTS
# --------------------------------------------------

def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )


# --------------------------------------------------
# 5. BUILD THE RAG CHATBOT
# --------------------------------------------------

def run_modern_bot():

    print("\n--- BUILDING THE BOT ---")

    # STEP 1: LOAD DOCUMENTS
    loader = TextLoader(
        "company_policy.txt",
        encoding="utf-8"
    )
    docs = loader.load()

    print(f"Loaded {len(docs)} document(s).")

    # STEP 2: SPLIT DOCUMENTS
    text_splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=1000,
        chunk_overlap=100
    )

    splits = text_splitter.split_documents(docs)

    print(f"Created {len(splits)} text chunk(s).")

    # STEP 3: CREATE EMBEDDINGS
    print("Creating embeddings...")

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    # STEP 4: CREATE FAISS VECTOR STORE
    vectorstore = FAISS.from_documents(
        documents=splits,
        embedding=embeddings
    )

    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 3}
    )

    print("Vector store created successfully.")

    # STEP 5: DEFINE THE PROMPT
    template = """
You are a helpful customer support assistant for TechStream.

Answer the question using only the retrieved context.
If the answer is not available in the context, say:
"I don't know" or "That is not in our policy."

Do not invent company policies.

Context:
{context}

Question:
{question}

Answer:
"""

    prompt = ChatPromptTemplate.from_template(template)

    # STEP 6: INITIALIZE GROQ LLM
    llm = ChatGroq(
    model= "openai/gpt-oss-20b",
    temperature=0
    )

    # STEP 7: BUILD THE LCEL RAG CHAIN
    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    print("\nBot is online!")

    # STEP 8: TEST THE CHATBOT
    questions = [
        "What is the return policy for laptops?",
        "Do you sell hamburgers?",
        "What are your support hours?"
    ]

    for q in questions:
        print(f"\nUser: {q}")

        try:
            response = rag_chain.invoke(q)
            print(f"Bot: {response}")

        except Exception as e:
            print(f"Error processing question: {e}")


# --------------------------------------------------
# 6. RUN THE PROGRAM
# --------------------------------------------------

if __name__ == "__main__":
    run_modern_bot()


Groq API key loaded successfully!
Libraries imported successfully.
Knowledge base created: company_policy.txt

--- BUILDING THE BOT ---
Loaded 1 document(s).
Created 1 text chunk(s).
Creating embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store created successfully.

Bot is online!

User: What is the return policy for laptops?
Bot: The return policy for laptops is a 14‑day window from delivery, but the laptop must be returned **unopened with the original factory seal intact**. If the seal on the laptop box is broken, we cannot accept the return due to security and data‑privacy reasons.

User: Do you sell hamburgers?
Bot: I don't know.

User: What are your support hours?
Bot: Our technical support team is available Monday to Friday, from 9:00 AM to 5:00 PM EST. We are closed on weekends and federal holidays.


**-------------END-------------**